# HMK / 02 — Longer windows for the intraday chains: 1 h on 40 / 60 bars, 15 m on 60 / 80 bars; set bands vs HMM (descriptive)

Protocol: [`HMK 004`](../../discussions/MNQ_Hierarchical_Markov/004_claude_hmk02_longer_windows.md). Code: `src/equity_trading/hmk.py`. **No trading test.**

- **Daily reference:** 20 days, ±5% (as in 01).
- **Window return** S_t = sum of the last W bar log returns (bar t included): 1 h W = 40, 60; 15 m W = 60, 80.
- **(a) Set band:** the daily ±5% / 20 days scaled by √time to W bars: 1 h ±1.47% (40), ±1.81% (60); 15 m ±0.90% (60), ±1.04% (80).
- **(b) HMM:** 3-state Gaussian HMM on S_t, fitted on the window; states = argmax of the causal filtered probabilities. The HMM's state means are the bull / bear returns it 'expects'; the midpoints between them are its implied thresholds.
- **Per case:** state shares, counts, P, persistence vs shuffled returns (same W, same rule or refitted HMM) and vs non-overlapping W-bar blocks, P^2/3/5/20, stationary distribution, run lengths, next-bar up share by state (descriptive).
- **Window:** 2020-01 – 2023-06 and halves 2020-01 – 2021-09 / 2021-10 – 2023-06.

Environment: `HMK_SYNTHETIC=1` (random-walk smoke run), `HMK_SYNTH_DIR`.

In [ ]:
import json, os, sys, tempfile, warnings
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)
SYNTHETIC = os.environ.get("HMK_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("HMK_SYNTH_DIR") or tempfile.mkdtemp(prefix="hmk_")
    os.environ["HMK_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import hmk as H
from equity_trading import micro_bars as MB

D = dt.date
CFG = H.HMKConfig()
if SYNTHETIC:
    WINDOWS = {"all": (D(2020, 1, 1), D(2021, 6, 30)), "H1": (D(2020, 1, 1), D(2020, 9, 30)), "H2": (D(2020, 10, 1), D(2021, 6, 30))}
    if not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
        H.write_synthetic(os.environ["EQUITY_TRADING_DIR"], CFG.load_start, WINDOWS["all"][1], effect=0.0)
    N_NULL_RULE, N_NULL_HMM = 30, 3
else:
    WINDOWS = {"all": (D(2020, 1, 1), D(2023, 6, 30)), "H1": (D(2020, 1, 1), D(2021, 9, 30)), "H2": (D(2021, 10, 1), D(2023, 6, 30))}
    N_NULL_RULE, N_NULL_HMM = 200, 20
CASES = [("1d", 20)] + [("1h", w) for w in (40, 60)] + [("15m", w) for w in (60, 80)]
OUT = Path("02"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 50)
print("engine", H.HMK_VERSION, "| synthetic random walk" if SYNTHETIC else "| real data")
print({f"{t} W={w}": f"±{100 * H.band_for_window(t, w):.2f}%" for t, w in CASES})

In [ ]:
m1 = MB.load_1m(CFG.root, CFG.load_start, WINDOWS["all"][1] + dt.timedelta(days=1))
BARS = H.session_bars(m1, CFG)
del m1
def series(tier, a, b):
    x = BARS[tier]
    t = pd.to_datetime(x.trade_date).dt.date
    return x[t <= b].ret_bp.to_numpy(float), (t[t <= b] >= a).to_numpy()

REP = {}
for tier, w in CASES:
    for win in WINDOWS:
        r, inwin = series(tier, *WINDOWS[win])
        full = win == "all"
        REP[(tier, w, "band", win)] = H.chain_report(r, w, H.band_for_window(tier, w), n_null=N_NULL_RULE if full else 0, mask=inwin)
        if tier != "1d":
            REP[(tier, w, "hmm", win)] = H.hmm_report(r, w, inwin, CFG, n_null=N_NULL_HMM if full else 0)
    print(tier, w, "done", flush=True)

In [ ]:
SUM = []
for (t, w, kind, win), r in REP.items():
    for s in H.STATE_NAMES:
        SUM.append({"tier": t, "W": w, "states": kind, "window": win, "state": s, "n_bars": r["n_bars"], "share": r["share"][s],
                    "p_stay": r["diag"][s], "p_stay_shuffled": r.get("null_shuffle_diag_mean", pd.Series(dtype=float)).get(s, np.nan),
                    "p_stay_shuffled_q95": r.get("null_shuffle_diag_q95", pd.Series(dtype=float)).get(s, np.nan),
                    "p_stay_nonoverlap": r["P_nonoverlap"].loc[s, s], "expected_duration_bars": r["expected_duration"][s],
                    "mean_run_bars": r["runs"].loc[s, "mean"], "stationary": r["stationary"][s],
                    "next_bar_up_share": r["next_bar_up_share"][s], "next_bar_n": r["next_bar_n"][s],
                    "band_pct": 100 * r["band"] if "band" in r else np.nan,
                    "hmm_mean_pct": r["state_mean_pct"][s] if "state_mean_pct" in r else np.nan,
                    "hmm_sd_pct": r["state_sd_pct"][s] if "state_sd_pct" in r else np.nan})
SUM = pd.DataFrame(SUM); SUM.to_csv(OUT / "02_summary.csv", index=False)
mats = []
for (t, w, kind, win), r in REP.items():
    for name, M in (("P", r["P"]), ("counts", r["counts"]), ("P_nonoverlap", r["P_nonoverlap"])) + tuple((f"P^{n}", m) for n, m in r["P_steps"].items()):
        mats.append(M.reset_index(names="from").assign(tier=t, W=w, states=kind, window=win, matrix=name))
pd.concat(mats).to_csv(OUT / "02_matrices.csv", index=False)
HMMP = {f"{t}_W{w}": {"mean_pct": r["state_mean_pct"].round(4).to_dict(), "sd_pct": r["state_sd_pct"].round(4).to_dict(),
                      "boundaries_pct": [round(b, 4) for b in r["boundaries_pct"]]}
        for (t, w, kind, win), r in REP.items() if kind == "hmm" and win == "all"}
json.dump(HMMP, open(OUT / "02_hmm_states.json", "w"), indent=1)
print(json.dumps(HMMP, indent=1))

## 1d, W = 20

In [ ]:
for kind in ('band',):
    r = REP[("1d", 20, kind, "all")]
    head = f"set band ±{100 * r['band']:.2f}%" if kind == "band" else f"HMM: state means {r['state_mean_pct'].round(3).to_dict()} %, boundaries {[round(b, 3) for b in r['boundaries_pct']]} %"
    print(f"=== 1d W=20 | {head} | bars {r['n_bars']} | non-overlapping blocks {r['n_blocks']}")
    print("state share:", r["share"].round(3).to_dict(), "| stationary:", r["stationary"].round(3).to_dict())
    print("counts"); display(r["counts"].astype(int))
    print("P(next bar's state | this bar's state)"); display(r["P"].round(3))
    print("persistence: diagonal vs shuffled-return reference vs non-overlapping blocks")
    display(pd.DataFrame({"p_stay": r["diag"], "shuffled_mean": r["null_shuffle_diag_mean"], "shuffled_q95": r["null_shuffle_diag_q95"],
                          "non_overlapping": np.diag(r["P_nonoverlap"]), "share": r["share"], "expected_duration": r["expected_duration"]}).round(3))
    print("P^2, P^3, P^5, P^20"); display(pd.concat({f"P^{n}": m for n, m in r["P_steps"].items()}).round(3))
    print("runs (bars)"); display(r["runs"].round(2))
    print("halves: p_stay / share"); display(SUM[(SUM.tier == "1d") & (SUM.W == 20) & (SUM.states == kind)].pivot_table(index="state", columns="window", values=["p_stay", "share"]).round(3))
    print("descriptive: share of next bars that close up, by state"); display(pd.DataFrame({"up_share": r["next_bar_up_share"], "n": r["next_bar_n"]}).round(4))

## 1h, W = 40

In [ ]:
for kind in ('band', 'hmm'):
    r = REP[("1h", 40, kind, "all")]
    head = f"set band ±{100 * r['band']:.2f}%" if kind == "band" else f"HMM: state means {r['state_mean_pct'].round(3).to_dict()} %, boundaries {[round(b, 3) for b in r['boundaries_pct']]} %"
    print(f"=== 1h W=40 | {head} | bars {r['n_bars']} | non-overlapping blocks {r['n_blocks']}")
    print("state share:", r["share"].round(3).to_dict(), "| stationary:", r["stationary"].round(3).to_dict())
    print("counts"); display(r["counts"].astype(int))
    print("P(next bar's state | this bar's state)"); display(r["P"].round(3))
    print("persistence: diagonal vs shuffled-return reference vs non-overlapping blocks")
    display(pd.DataFrame({"p_stay": r["diag"], "shuffled_mean": r["null_shuffle_diag_mean"], "shuffled_q95": r["null_shuffle_diag_q95"],
                          "non_overlapping": np.diag(r["P_nonoverlap"]), "share": r["share"], "expected_duration": r["expected_duration"]}).round(3))
    print("P^2, P^3, P^5, P^20"); display(pd.concat({f"P^{n}": m for n, m in r["P_steps"].items()}).round(3))
    print("runs (bars)"); display(r["runs"].round(2))
    print("halves: p_stay / share"); display(SUM[(SUM.tier == "1h") & (SUM.W == 40) & (SUM.states == kind)].pivot_table(index="state", columns="window", values=["p_stay", "share"]).round(3))
    print("descriptive: share of next bars that close up, by state"); display(pd.DataFrame({"up_share": r["next_bar_up_share"], "n": r["next_bar_n"]}).round(4))

## 1h, W = 60

In [ ]:
for kind in ('band', 'hmm'):
    r = REP[("1h", 60, kind, "all")]
    head = f"set band ±{100 * r['band']:.2f}%" if kind == "band" else f"HMM: state means {r['state_mean_pct'].round(3).to_dict()} %, boundaries {[round(b, 3) for b in r['boundaries_pct']]} %"
    print(f"=== 1h W=60 | {head} | bars {r['n_bars']} | non-overlapping blocks {r['n_blocks']}")
    print("state share:", r["share"].round(3).to_dict(), "| stationary:", r["stationary"].round(3).to_dict())
    print("counts"); display(r["counts"].astype(int))
    print("P(next bar's state | this bar's state)"); display(r["P"].round(3))
    print("persistence: diagonal vs shuffled-return reference vs non-overlapping blocks")
    display(pd.DataFrame({"p_stay": r["diag"], "shuffled_mean": r["null_shuffle_diag_mean"], "shuffled_q95": r["null_shuffle_diag_q95"],
                          "non_overlapping": np.diag(r["P_nonoverlap"]), "share": r["share"], "expected_duration": r["expected_duration"]}).round(3))
    print("P^2, P^3, P^5, P^20"); display(pd.concat({f"P^{n}": m for n, m in r["P_steps"].items()}).round(3))
    print("runs (bars)"); display(r["runs"].round(2))
    print("halves: p_stay / share"); display(SUM[(SUM.tier == "1h") & (SUM.W == 60) & (SUM.states == kind)].pivot_table(index="state", columns="window", values=["p_stay", "share"]).round(3))
    print("descriptive: share of next bars that close up, by state"); display(pd.DataFrame({"up_share": r["next_bar_up_share"], "n": r["next_bar_n"]}).round(4))

## 15m, W = 60

In [ ]:
for kind in ('band', 'hmm'):
    r = REP[("15m", 60, kind, "all")]
    head = f"set band ±{100 * r['band']:.2f}%" if kind == "band" else f"HMM: state means {r['state_mean_pct'].round(3).to_dict()} %, boundaries {[round(b, 3) for b in r['boundaries_pct']]} %"
    print(f"=== 15m W=60 | {head} | bars {r['n_bars']} | non-overlapping blocks {r['n_blocks']}")
    print("state share:", r["share"].round(3).to_dict(), "| stationary:", r["stationary"].round(3).to_dict())
    print("counts"); display(r["counts"].astype(int))
    print("P(next bar's state | this bar's state)"); display(r["P"].round(3))
    print("persistence: diagonal vs shuffled-return reference vs non-overlapping blocks")
    display(pd.DataFrame({"p_stay": r["diag"], "shuffled_mean": r["null_shuffle_diag_mean"], "shuffled_q95": r["null_shuffle_diag_q95"],
                          "non_overlapping": np.diag(r["P_nonoverlap"]), "share": r["share"], "expected_duration": r["expected_duration"]}).round(3))
    print("P^2, P^3, P^5, P^20"); display(pd.concat({f"P^{n}": m for n, m in r["P_steps"].items()}).round(3))
    print("runs (bars)"); display(r["runs"].round(2))
    print("halves: p_stay / share"); display(SUM[(SUM.tier == "15m") & (SUM.W == 60) & (SUM.states == kind)].pivot_table(index="state", columns="window", values=["p_stay", "share"]).round(3))
    print("descriptive: share of next bars that close up, by state"); display(pd.DataFrame({"up_share": r["next_bar_up_share"], "n": r["next_bar_n"]}).round(4))

## 15m, W = 80

In [ ]:
for kind in ('band', 'hmm'):
    r = REP[("15m", 80, kind, "all")]
    head = f"set band ±{100 * r['band']:.2f}%" if kind == "band" else f"HMM: state means {r['state_mean_pct'].round(3).to_dict()} %, boundaries {[round(b, 3) for b in r['boundaries_pct']]} %"
    print(f"=== 15m W=80 | {head} | bars {r['n_bars']} | non-overlapping blocks {r['n_blocks']}")
    print("state share:", r["share"].round(3).to_dict(), "| stationary:", r["stationary"].round(3).to_dict())
    print("counts"); display(r["counts"].astype(int))
    print("P(next bar's state | this bar's state)"); display(r["P"].round(3))
    print("persistence: diagonal vs shuffled-return reference vs non-overlapping blocks")
    display(pd.DataFrame({"p_stay": r["diag"], "shuffled_mean": r["null_shuffle_diag_mean"], "shuffled_q95": r["null_shuffle_diag_q95"],
                          "non_overlapping": np.diag(r["P_nonoverlap"]), "share": r["share"], "expected_duration": r["expected_duration"]}).round(3))
    print("P^2, P^3, P^5, P^20"); display(pd.concat({f"P^{n}": m for n, m in r["P_steps"].items()}).round(3))
    print("runs (bars)"); display(r["runs"].round(2))
    print("halves: p_stay / share"); display(SUM[(SUM.tier == "15m") & (SUM.W == 80) & (SUM.states == kind)].pivot_table(index="state", columns="window", values=["p_stay", "share"]).round(3))
    print("descriptive: share of next bars that close up, by state"); display(pd.DataFrame({"up_share": r["next_bar_up_share"], "n": r["next_bar_n"]}).round(4))

## Summary and heatmaps

In [ ]:
display(SUM[SUM.window == "all"].pivot_table(index=["tier", "W", "states", "state"],
        values=["share", "p_stay", "p_stay_shuffled", "p_stay_nonoverlap", "next_bar_up_share", "hmm_mean_pct"], sort=False).round(3))
keys = [k for k in REP if k[3] == "all"]
fig, ax = plt.subplots(2, len(keys), figsize=(3.6 * len(keys), 7.5))
for j, k in enumerate(keys):
    for i, (name, M) in enumerate((("overlapping", REP[k]["P"]), ("non-overlapping", REP[k]["P_nonoverlap"]))):
        a = ax[i, j]; v = M.to_numpy(float); a.imshow(v, vmin=0, vmax=1, cmap="Blues")
        for (p, q), x in np.ndenumerate(v):
            a.text(q, p, f"{x:.2f}" if np.isfinite(x) else "-", ha="center", va="center", color="k" if not np.isfinite(x) or x < 0.6 else "w", fontsize=8)
        a.set_xticks(range(3), H.STATE_NAMES, fontsize=8); a.set_yticks(range(3), H.STATE_NAMES, fontsize=8)
        a.set_title(f"{k[0]} W={k[1]} {k[2]} — {name}", fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "02_matrices.png", dpi=110); plt.show()
json.dump({"version": H.HMK_VERSION, "synthetic": SYNTHETIC, "windows": {k: [str(a), str(b)] for k, (a, b) in WINDOWS.items()},
           "cases": CASES, "bands_pct": {f"{t}_W{w}": 100 * H.band_for_window(t, w) for t, w in CASES},
           "n_null_rule": N_NULL_RULE, "n_null_hmm": N_NULL_HMM}, open(OUT / "02_run.json", "w"), indent=1)